# Modelo: SVD (Filtrado Colaborativo)

Matrix factorization con TruncatedSVD de scikit-learn.
Dado la matriz usuario-restaurante con ratings, la factoriza en
factores latentes y predice ratings no observados.

**Hiperparámetros:** k=50 factores latentes, random_state=42.

Referencia extraida de https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html

In [6]:
import os
import json
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
import sys
sys.path.append('..')
from src.evaluation import temporal_train_test_split, evaluate_model

MODEL_NAME = 'svd'
RESULTS_DIR = f'../results/{MODEL_NAME}'
N_FACTORS = 50

## Definición del modelo

In [ ]:
class SVDRecommender:
    """
    Filtrado colaborativo mediante TruncatedSVD (scikit-learn).
    Construye la matriz usuario-ítem con ratings y la factoriza con
    randomized SVD. La predicción es U_sigma @ Vt donde U_sigma = U @ diag(sigma).

    Usa randomized SVD, más rápido en matrices grandes.
    """
    def __init__(self, n_factors=50, random_state=42):
        self.n_factors = n_factors
        self.random_state = random_state
        self._user_index = {}
        self._item_index = {}
        self._U_sigma = None
        self._Vt = None
        self._predicted = None

    def fit(self, reviews_df):
        users = reviews_df['user_id'].unique()
        items = reviews_df['business_id'].unique()
        self._user_index = {u: i for i, u in enumerate(users)}
        self._item_index = {b: i for i, b in enumerate(items)}

        rows = reviews_df['user_id'].map(self._user_index)
        cols = reviews_df['business_id'].map(self._item_index)
        vals = reviews_df['stars'].astype(float)
        matrix = csr_matrix((vals, (rows, cols)), shape=(len(users), len(items)))

        svd = TruncatedSVD(n_components=self.n_factors, random_state=self.random_state)
        self._U_sigma = svd.fit_transform(matrix)
        self._Vt = svd.components_
        self._predicted = self._U_sigma @ self._Vt

        print(f'SVD fitted: {len(users)} users x {len(items)} items, k={self.n_factors}')
        return self

    def recommend(self, user_id, train_reviews, top_k=10):
        if user_id not in self._user_index:
            return []
        u_idx = self._user_index[user_id]
        scores = self._predicted[u_idx].copy()
        seen = set(train_reviews[train_reviews['user_id'] == user_id]['business_id'])
        idx_to_item = {v: k for k, v in self._item_index.items()}
        for bid, b_idx in self._item_index.items():
            if bid in seen:
                scores[b_idx] = -np.inf
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [idx_to_item[i] for i in top_indices if i in idx_to_item]

## Datos

In [8]:
reviews = pd.read_csv('../data/processed/reviews.csv', parse_dates=['date'])
train_reviews, test_reviews = temporal_train_test_split(reviews, test_fraction=0.2)
print(f'Train: {len(train_reviews)} | Test: {len(test_reviews)}')

Train: 83256 reviews | Test: 17191 reviews
Train: 83256 | Test: 17191


## Entrenamiento y evaluación

In [9]:
model = SVDRecommender(n_factors=N_FACTORS).fit(train_reviews)

metrics = evaluate_model(
    lambda uid, top_k: model.recommend(uid, train_reviews, top_k),
    test_reviews, train_reviews, k_values=[5, 10, 20]
)
print(metrics.round(4))

SVD fitted: 10490 users x 1151 items, k=50
    precision  recall    ndcg
K                            
5      0.0239  0.0789  0.0562
10     0.0204  0.1326  0.0749
20     0.0163  0.2049  0.0953


## Guardar resultados

In [10]:
os.makedirs(RESULTS_DIR, exist_ok=True)
metrics.to_csv(f'{RESULTS_DIR}/metrics.csv')
with open(f'{RESULTS_DIR}/config.json', 'w') as f:
    json.dump({'model': 'SVD', 'n_factors': N_FACTORS, 'random_state': 42,
               'library': 'sklearn.decomposition.TruncatedSVD'}, f, indent=2)
print(f'Saved -> results/{MODEL_NAME}/')

Saved -> results/svd/
